# Phase 4b — SetFit : Sentence Transformer + Contrastive Learning

## Pourquoi SetFit au lieu de DistilBERT classique ?

DistilBERT + Linear head est un **classifieur fermé** — il force toujours une prédiction
parmi vos 66 intents, même pour une phrase hors domaine. C'est pourquoi Banking77/CLINC150
donnent des scores effondrés : le modèle ne peut pas dire 'je ne sais pas'.

**SetFit fonctionne différemment :**

```
Étape 1 — Fine-tuning contrastif
   Paires (phrase_A, phrase_B) → label 1 si même intent, 0 sinon
   Le Sentence Transformer apprend que 'I want a refund' ≈ 'give me my money back'
   dans l'espace sémantique, indépendamment des mots exacts.

Étape 2 — Tête de classification
   Les embeddings produits alimentent un classifieur léger (LogisticRegression)
   Le modèle compare des vecteurs de sens, pas des patterns de mots.
```

**Avantage clé pour votre projet :**
Le Sentence Transformer a été pré-entraîné sur des milliards de paires de phrases
du web — il comprend déjà que 'wtf still no delivery' et 'where is my package' sont
sémantiquement proches. Le fine-tuning contrastif affine cette compréhension pour
votre domaine e-commerce spécifiquement.

**Ce notebook est un test rapide — objectif : valider la convergence en < 1h**

| Paramètre | Valeur test | Valeur production |
|-----------|------------|------------------|
| Échantillon train | 8 exemples/intent | tous |
| Epochs contrastif | 1 | 3-5 |
| Backbone | all-MiniLM-L6-v2 | paraphrase-mpnet-base-v2 |

**Référence :** Tunstall et al. — SetFit: Efficient Few-Shot Learning Without Prompts — NeurIPS 2022


## 0. Installation (à faire une seule fois)

In [5]:
# Décommenter et lancer si pas encore installé
import torch
import setfit
import sentence_transformers
from transformers.training_args import default_logdir
print(f'SetFit version          : {setfit.__version__}')
print(f'SentenceTransformers    : {sentence_transformers.__version__}')


SetFit version          : 0.7.0
SentenceTransformers    : 2.5.1


## 1. Imports & Configuration

In [6]:
import json
import time
import numpy as np
import pandas as pd
from pathlib import Path

from datasets import Dataset
from setfit import SetFitModel, SetFitTrainer
from setfit.data import sample_dataset
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch

# ── Chemins ──────────────────────────────────────────────────
BASE_DIR    = Path(r'D:/conv_nlp_pipeline')
SPLITS_DIR  = BASE_DIR / 'data' / 'splits'
PROC_DIR    = BASE_DIR / 'data' / 'processed'
RESULTS_DIR = BASE_DIR / 'results' / 'setfit'
MODEL_DIR   = BASE_DIR / 'models' / 'setfit'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
TEXT_COL    = 'instruction_clean'
LABEL_COL   = 'label'

# ── Config test rapide (<1h) ─────────────────────────────────
# Changer ces valeurs pour la version production
NUM_SAMPLES_PER_CLASS = 8    
NUM_EPOCHS_CONTRASTIVE = 3   
BACKBONE = 'sentence-transformers/all-MiniLM-L6-v2'
           # rapide : all-MiniLM-L6-v2 (80MB)
           # production : paraphrase-mpnet-base-v2 (420MB)

print(f'Device : {"GPU" if torch.cuda.is_available() else "CPU"}')
print(f'Backbone : {BACKBONE}')
print(f'Samples/class : {NUM_SAMPLES_PER_CLASS}')
print(f'Mode : TEST RAPIDE — changer NUM_SAMPLES_PER_CLASS="all" pour production')


Device : CPU
Backbone : sentence-transformers/all-MiniLM-L6-v2
Samples/class : 8
Mode : TEST RAPIDE — changer NUM_SAMPLES_PER_CLASS="all" pour production


## 2. Chargement des splits (produits par le notebook EDA)

In [7]:
# Ces fichiers viennent directement de 02_EDA_Preprocessing
# Le notebook EDA ne change pas — SetFit consomme les mêmes splits
train_df = pd.read_csv(SPLITS_DIR / 'train.csv')
val_df   = pd.read_csv(SPLITS_DIR / 'val.csv')
test_df  = pd.read_csv(SPLITS_DIR / 'test.csv')

with open(PROC_DIR / 'label_map.json', 'r', encoding='utf-8') as f:
    LABEL2ID = json.load(f)
ID2LABEL   = {int(v): k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)

# Nettoyage minimal
for df_ in [train_df, val_df, test_df]:
    df_[TEXT_COL]  = df_[TEXT_COL].fillna('').astype(str)
    df_[LABEL_COL] = df_[LABEL_COL].astype(int)

print(f'Train : {len(train_df):,}  |  Val : {len(val_df):,}  |  Test : {len(test_df):,}')
print(f'Intents : {NUM_LABELS}')

if 'source' in train_df.columns:
    print('\nComposition du train :')
    print(train_df['source'].value_counts().to_string())


Train : 83,156  |  Val : 6,655  |  Test : 6,672
Intents : 66

Composition du train :
source
retail                    31152
support                   24533
augmented_easy            11137
augmented_hard             8167
amazon_single_qna          6918
amazon_multi_questions      909
synthetic_ecommerce         288
ecommerce_faq                52


## 3. Conversion en HuggingFace Dataset

SetFit attend un `Dataset` HuggingFace avec colonnes `text` et `label`.


In [8]:
def to_hf(df_):
    return Dataset.from_dict({
        'text':  df_[TEXT_COL].tolist(),
        'label': df_[LABEL_COL].tolist(),
    })

ds_train_full = to_hf(train_df)
ds_val        = to_hf(val_df)
ds_test       = to_hf(test_df)

# ── Sous-échantillonner pour le test rapide ───────────────────
# sample_dataset garde exactement NUM_SAMPLES_PER_CLASS exemples par intent
# En production : supprimer cette ligne et utiliser ds_train_full
if NUM_SAMPLES_PER_CLASS != 'all':
    ds_train = sample_dataset(
        ds_train_full,
        label_column='label',
        num_samples=NUM_SAMPLES_PER_CLASS,
        seed=RANDOM_SEED
    )
    print(f'Train sous-échantillonné : {len(ds_train)} exemples')
    print(f'  ({NUM_SAMPLES_PER_CLASS} exemples × {NUM_LABELS} intents = {NUM_SAMPLES_PER_CLASS*NUM_LABELS})')
else:
    ds_train = ds_train_full
    print(f'Train complet : {len(ds_train)} exemples')


Train sous-échantillonné : 528 exemples
  (8 exemples × 66 intents = 528)


## 4. Chargement du modèle SetFit

### Comment fonctionne le backbone ?

`all-MiniLM-L6-v2` est un Sentence Transformer pré-entraîné sur 1 milliard de paires
de phrases du web. Il transforme n'importe quelle phrase en un vecteur de 384 dimensions
qui encode son **sens** — pas ses mots.

```
'I want a refund'        → [0.23, -0.45, 0.12, ...]  ← 384 dimensions
'give me my money back'  → [0.21, -0.43, 0.14, ...]  ← très proche du premier
'where is my package'    → [-0.34, 0.67, -0.23, ...]  ← éloigné
```

Le fine-tuning contrastif SetFit va **resserrer** les embeddings du même intent
et **écarter** ceux d'intents différents, sur votre domaine e-commerce.


In [9]:
model = SetFitModel.from_pretrained(
    BACKBONE,
    labels=list(LABEL2ID.keys()),
    # id2label et label2id sont inférés automatiquement depuis labels
)

print(f'Modèle chargé : {BACKBONE}')
print(f'Embedding dim : {model.model_body.get_sentence_embedding_dimension()}')
print(f'Nombre de labels : {NUM_LABELS}')
print(f'Tête classifieur : {type(model.model_head).__name__}')

# Démonstration avant fine-tuning
test_phrases = [
    'I want a refund for my order',
    'give me my money back NOW',
    'where is my package??',
    'still waiting 4 my order wtf',
]
print('\nEmbeddings avant fine-tuning (similarité cosinus) :')
from sentence_transformers import util
embs = model.model_body.encode(test_phrases, convert_to_tensor=True)
for i in range(len(test_phrases)):
    for j in range(i+1, len(test_phrases)):
        sim = util.cos_sim(embs[i], embs[j]).item()
        print(f'  sim({i},{j}) = {sim:.3f}  |  "{test_phrases[i][:30]}" <-> "{test_phrases[j][:30]}"')


config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

`SentenceTransformer._target_device` has been removed, please use `SentenceTransformer.device` instead.
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


Modèle chargé : sentence-transformers/all-MiniLM-L6-v2
Embedding dim : 384
Nombre de labels : 66
Tête classifieur : LogisticRegression

Embeddings avant fine-tuning (similarité cosinus) :
  sim(0,1) = 0.446  |  "I want a refund for my order" <-> "give me my money back NOW"
  sim(0,2) = 0.266  |  "I want a refund for my order" <-> "where is my package??"
  sim(0,3) = 0.454  |  "I want a refund for my order" <-> "still waiting 4 my order wtf"
  sim(1,2) = 0.108  |  "give me my money back NOW" <-> "where is my package??"
  sim(1,3) = 0.324  |  "give me my money back NOW" <-> "still waiting 4 my order wtf"
  sim(2,3) = 0.348  |  "where is my package??" <-> "still waiting 4 my order wtf"


## 5. Configuration de l'entraînement

SetFit entraîne en **deux phases séquentielles** :

**Phase contrastive** — le Sentence Transformer apprend à rapprocher
les phrases du même intent et à éloigner celles d'intents différents.
Duration : `num_epochs × num_iterations` paires.

**Phase classifieur** — sur les embeddings figés, entraîne une
LogisticRegression en quelques secondes.


In [14]:
from setfit import SetFitModel, SetFitTrainer
from sklearn.metrics import f1_score, accuracy_score
import random, numpy as np, torch

# Reproductibilité
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Modèle
model = SetFitModel.from_pretrained(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

# Métriques
def compute_metrics(y_pred, y_test):
    return {
        "f1": f1_score(y_test, y_pred, average="macro", zero_division=0),
        "accuracy": accuracy_score(y_test, y_pred),
    }

# Trainer compatible
trainer = SetFitTrainer(
    model=model,
    train_dataset=ds_train,
    eval_dataset=ds_val,

    # Phase contrastive
    num_epochs=NUM_EPOCHS_CONTRASTIVE,
    num_iterations=5,
    batch_size=16,

    # Mapping colonnes
    column_mapping={"text": "text", "label": "label"},

    # Métriques
    metric=compute_metrics,
)

print("SetFitTrainer OK ✓")

# Entraînement
trainer.train()

# Évaluation
metrics = trainer.evaluate()
print(metrics)

# Sauvegarde

OUTPUT_MODEL_DIR = "outputs/setfit_model"
model.save_pretrained(OUTPUT_MODEL_DIR)

print(f"Modèle sauvegardé dans : {OUTPUT_MODEL_DIR}")

`SentenceTransformer._target_device` has been removed, please use `SentenceTransformer.device` instead.
model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.
Applying column mapping to training dataset


SetFitTrainer OK ✓


Generating Training Pairs:   0%|          | 0/5 [00:00<?, ?it/s]

***** Running training *****
  Num examples = 5280
  Num epochs = 1
  Total optimization steps = 330
  Total train batch size = 16


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/330 [00:00<?, ?it/s]

Applying column mapping to evaluation dataset
***** Running evaluation *****


{'f1': 0.680154898489153, 'accuracy': 0.8805409466566492}
Modèle sauvegardé dans : outputs/setfit_model


## 6. Initialisation du Trainer

## 7. Entraînement

> **Durée estimée — mode test rapide (8 exemples/intent, 1 epoch) :**
> - GPU : ~5–15 minutes
> - CPU : ~20–40 minutes

> **Durée estimée — mode production (tous exemples, 3 epochs) :**
> - GPU : ~30–60 minutes
> - CPU : plusieurs heures


In [15]:
print('Début de l\'entraînement SetFit...')
print(f'  Backbone     : {BACKBONE}')
print(f'  Train size   : {len(ds_train)}')
print(f'  Epochs       : {NUM_EPOCHS_CONTRASTIVE}')
print()

t0 = time.time()
trainer.train()
elapsed = time.time() - t0

print(f'\nEntraînement terminé en {elapsed:.0f}s ({elapsed/60:.1f} min)')


Applying column mapping to training dataset


Début de l'entraînement SetFit...
  Backbone     : sentence-transformers/all-MiniLM-L6-v2
  Train size   : 528
  Epochs       : 1



Generating Training Pairs:   0%|          | 0/5 [00:00<?, ?it/s]

***** Running training *****
  Num examples = 5280
  Num epochs = 1
  Total optimization steps = 330
  Total train batch size = 16


Epoch:   0%|          | 0/1 [00:00<?, ?it/s]

Iteration:   0%|          | 0/330 [00:00<?, ?it/s]


Entraînement terminé en 470s (7.8 min)


## 8. Évaluation — Test Retail (in-domain)

In [24]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch
import torch

print("\n" + "="*60)
print("DEBUG — VERIFICATION AVANT METRICS")
print("="*60)

# 1. Taille
print(f"Nb samples labels : {len(retail_labels)}")
print(f"Nb samples preds  : {len(retail_preds_names)}")

# 2. Types
print("\nTypes :")
print("Type label réel   :", type(retail_labels[0]))
print("Type prédiction   :", type(retail_preds_names[0]))

# 3. Exemples
print("\nExemples :")
for i in range(5):
    print(f"{i} | TRUE = {retail_labels[i]} | PRED = {retail_preds_names[i]}")

# 4. Valeurs uniques (important)
print("\nValeurs uniques (top 10) :")
print("Labels réels :", list(set(retail_labels))[:10])
print("Predictions  :", list(set(retail_preds_names))[:10])

# 5. Vérif mapping (si tu utilises LABEL2ID)
if 'LABEL2ID' in globals():
    missing = [p for p in retail_preds_names if str(p) not in LABEL2ID]
    print(f"\nLabels prédits NON trouvés dans LABEL2ID : {len(missing)}")
    print("Exemples :", missing[:10])

# 6. Vérif tensors (ERREUR fréquente)
tensor_count_true = sum(1 for x in retail_labels if isinstance(x, torch.Tensor))
tensor_count_pred = sum(1 for x in retail_preds_names if isinstance(x, torch.Tensor))

print("\nTensors détectés :")
print("Labels réels :", tensor_count_true)
print("Predictions  :", tensor_count_pred)

print("="*60)
def clean_label(x):
    if isinstance(x, torch.Tensor):
        return int(x.item())
    return int(x)

from sklearn.metrics import f1_score, accuracy_score, classification_report
import torch

# Textes + vrais labels
retail_texts = test_df[TEXT_COL].tolist()
retail_labels = [int(x) for x in test_df[LABEL_COL].tolist()]

# Prédictions SetFit
retail_preds_raw = model.predict(retail_texts)

# Conversion Tensor -> int
retail_preds = [
    int(p.item()) if isinstance(p, torch.Tensor) else int(p)
    for p in retail_preds_raw
]

# Vérification
print(type(retail_labels[0]), retail_labels[:10])
print(type(retail_preds[0]), retail_preds[:10])

# Métriques
retail_f1 = f1_score(retail_labels, retail_preds, average="macro", zero_division=0)
retail_acc = accuracy_score(retail_labels, retail_preds)

print("=" * 55)
print("  TEST RETAIL — In-domain")
print("=" * 55)
print(f"  F1 macro  : {retail_f1:.4f}")
print(f"  Accuracy  : {retail_acc:.4f}")
print("=" * 55)

labels_present = sorted(set(retail_labels) | set(retail_preds))

print(classification_report(
    retail_labels,
    retail_preds,
    labels=labels_present,
    digits=3,
    zero_division=0
))


DEBUG — VERIFICATION AVANT METRICS
Nb samples labels : 6672
Nb samples preds  : 6672

Types :
Type label réel   : <class 'int'>
Type prédiction   : <class 'torch.Tensor'>

Exemples :
0 | TRUE = 2 | PRED = 2
1 | TRUE = 27 | PRED = 27
2 | TRUE = 61 | PRED = 61
3 | TRUE = 12 | PRED = 12
4 | TRUE = 57 | PRED = 50

Valeurs uniques (top 10) :
Labels réels : [0, 1, 2, 3, 4, 5, 6, 12, 17, 18]
Predictions  : [tensor(5, dtype=torch.int32), tensor(40, dtype=torch.int32), tensor(49, dtype=torch.int32), tensor(64, dtype=torch.int32), tensor(58, dtype=torch.int32), tensor(39, dtype=torch.int32), tensor(44, dtype=torch.int32), tensor(26, dtype=torch.int32), tensor(62, dtype=torch.int32), tensor(25, dtype=torch.int32)]

Labels prédits NON trouvés dans LABEL2ID : 6672
Exemples : [tensor(2, dtype=torch.int32), tensor(27, dtype=torch.int32), tensor(61, dtype=torch.int32), tensor(12, dtype=torch.int32), tensor(50, dtype=torch.int32), tensor(10, dtype=torch.int32), tensor(3, dtype=torch.int32), tensor(46,

## 9. Scores de confiance — Détection OOD

SetFit expose les probabilités de la tête LogisticRegression.
Un score de confiance bas signifie que la phrase est loin de tous les prototypes d'intents
— c'est votre détecteur OOD naturel.


In [17]:
import torch.nn.functional as F

def predict_with_confidence(texts, batch_size=64):
    """
    Retourne (preds_str, confidences).
    Utilise les probabilités de la tête classifieur SetFit.
    """
    all_preds, all_confs = [], []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        # Embeddings via Sentence Transformer
        embeddings = model.model_body.encode(
            batch, convert_to_tensor=True, show_progress_bar=False
        )

        # Probabilités via tête classifieur
        if hasattr(model.model_head, 'predict_proba'):
            # LogisticRegression sklearn
            proba = model.model_head.predict_proba(embeddings.cpu().numpy())
            conf  = proba.max(axis=1)
            preds = proba.argmax(axis=1)
            preds_str = [str(model.labels_[p]) if hasattr(model, 'labels_') else ID2LABEL.get(p, str(p)) for p in preds]
        else:
            # SetFitHead torch
            with torch.no_grad():
                logits = model.model_head(embeddings)
                proba  = F.softmax(logits, dim=-1).cpu().numpy()
            conf  = proba.max(axis=1)
            preds = proba.argmax(axis=1)
            preds_str = [ID2LABEL.get(int(p), str(p)) for p in preds]

        all_preds.extend(preds_str)
        all_confs.extend(conf.tolist())

    return all_preds, all_confs

# Confiance sur Retail (in-domain)
_, confs_retail = predict_with_confidence(retail_texts)
print(f'Confiance moyenne Retail (in-domain) : {np.mean(confs_retail):.4f}')


Confiance moyenne Retail (in-domain) : 0.7800


## 10. Test Externe — CLINC150 + Banking77

Ici on ne fait **pas de mapping forcé** — on mesure directement :
1. La confiance du modèle sur des phrases hors domaine
2. Le F1 uniquement sur les intents clairement mappables (labels humains)

C'est la mesure de robustesse honnête.


In [18]:
from datasets import load_dataset

# ── CLINC150 ─────────────────────────────────────────────────
CLINC_TO_BITEXT = {
    'order_status':     'track_order',
    'cancel_order':     'cancel_order',
    'return':           'return_product',
    'refund':           'request_refund',
    'pay_bill':         'pay',
    'card_arrival':     'track_delivery',
    'shipping':         'shipping_costs',
    'damaged_item':     'damaged_delivery',
    'missing_item':     'missing_item',
    'wrong_item':       'wrong_item',
    'change_password':  'recover_password',
    'freeze_account':   'close_account',
    'find_store':       'store_location',
    'store_hours':      'store_opening_hours',
    'complaint':        'complaint',
    'review':           'review',
    'shopping_list':    'add_product',
}

ext_texts, ext_intents = [], []

try:
    clinc = load_dataset('clinc_oos', 'plus')
    clinc_names = clinc['test'].features['intent'].names
    df_clinc = clinc['test'].to_pandas()
    df_clinc['clinc_intent'] = df_clinc['intent'].map(lambda x: clinc_names[x])
    df_clinc_m = df_clinc[df_clinc['clinc_intent'].isin(CLINC_TO_BITEXT)].copy()
    df_clinc_m['bitext_intent'] = df_clinc_m['clinc_intent'].map(CLINC_TO_BITEXT)
    df_clinc_m = df_clinc_m[df_clinc_m['bitext_intent'].isin(LABEL2ID)]
    ext_texts.extend(df_clinc_m['text'].tolist())
    ext_intents.extend(df_clinc_m['bitext_intent'].tolist())
    print(f'CLINC150 : {len(df_clinc_m)} exemples mappés · {df_clinc_m["bitext_intent"].nunique()} intents')
except Exception as e:
    print(f'CLINC150 : {e}')

# ── Banking77 ─────────────────────────────────────────────────
BANKING_TO_BITEXT = {
    'cancel_transfer':       'cancel_order',
    'card_arrival':          'track_delivery',
    'refund_not_showing_up': 'refund_status',
    'request_refund':        'request_refund',
    'pending_transfer':      'track_order',
    'transaction_charged_twice': 'complaint',
    'wrong_exchange_rate_for_cash_withdrawal': 'payment_issue',
    'passcode_forgotten':    'recover_password',
    'lost_or_stolen_card':   'customer_service',
    'edit_personal_details': 'edit_account',
    'terminate_account':     'close_account',
    'dispute_purchase':      'complaint',
}

try:
    banking = load_dataset('PolyAI/banking77')
    banking_names = banking['test'].features['label'].names
    df_b = banking['test'].to_pandas()
    df_b['banking_intent'] = df_b['label'].map(lambda x: banking_names[x])
    df_b_m = df_b[df_b['banking_intent'].isin(BANKING_TO_BITEXT)].copy()
    df_b_m['bitext_intent'] = df_b_m['banking_intent'].map(BANKING_TO_BITEXT)
    df_b_m = df_b_m[df_b_m['bitext_intent'].isin(LABEL2ID)]
    ext_texts.extend(df_b_m['text'].tolist())
    ext_intents.extend(df_b_m['bitext_intent'].tolist())
    print(f'Banking77 : {len(df_b_m)} exemples mappés · {df_b_m["bitext_intent"].nunique()} intents')
except Exception as e:
    print(f'Banking77 : {e}')

print(f'\nTotal test externe : {len(ext_texts)} exemples')


CLINC150 : 120 exemples mappés · 4 intents


D:\conv_nlp_pipeline\venv\lib\site-packages\datasets\load.py:1429: FutureWarning: The repository for PolyAI/banking77 contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/PolyAI/banking77
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


Banking77 : 400 exemples mappés · 10 intents

Total test externe : 520 exemples


In [19]:
if len(ext_texts) > 0:
    ext_labels_ids = [LABEL2ID[i] for i in ext_intents]

    # Prédictions avec confiance
    ext_preds_names, ext_confs = predict_with_confidence(ext_texts)
    ext_preds_ids = [LABEL2ID.get(str(p), -1) for p in ext_preds_names]

    # F1 sur intents présents uniquement
    present_ids = list(set(ext_labels_ids))
    ext_f1  = f1_score(
        ext_labels_ids, ext_preds_ids,
        labels=present_ids, average='macro', zero_division=0
    )
    ext_acc = accuracy_score(ext_labels_ids, ext_preds_ids)

    # Confiance OOD (sur TOUTES les phrases Banking77 non mappées)
    try:
        all_banking_texts = banking['test'].to_pandas()['text'].tolist()
        _, ood_confs = predict_with_confidence(all_banking_texts)
        mean_ood_conf = np.mean(ood_confs)
    except:
        mean_ood_conf = np.mean(ext_confs)

    print('=' * 55)
    print('  TEST EXTERNE — CLINC150 + Banking77')
    print('=' * 55)
    print(f'  F1 macro (intents mappés)  : {ext_f1:.4f}')
    print(f'  Accuracy                   : {ext_acc:.4f}')
    print(f'  Confiance moy. (in-domain) : {np.mean(confs_retail):.4f}')
    print(f'  Confiance moy. (OOD)       : {mean_ood_conf:.4f}')
    print(f'  Gap OOD                    : {np.mean(confs_retail)-mean_ood_conf:+.4f}')
    print('=' * 55)
    print()

    # Par intent mappé
    present_names = [ID2LABEL[i] for i in sorted(set(present_ids))]
    print('F1 par intent (test externe) :')
    print(classification_report(
        ext_labels_ids, ext_preds_ids,
        labels=sorted(set(present_ids)),
        target_names=present_names,
        digits=3, zero_division=0
    ))


  TEST EXTERNE — CLINC150 + Banking77
  F1 macro (intents mappés)  : 0.3333
  Accuracy                   : 0.2942
  Confiance moy. (in-domain) : 0.7800
  Confiance moy. (OOD)       : 0.3149
  Gap OOD                    : +0.4651

F1 par intent (test externe) :
                  precision    recall  f1-score   support

     add_product      1.000     0.067     0.125        30
    cancel_order      0.609     0.350     0.444        40
   close_account      0.854     0.500     0.631        70
       complaint      1.000     0.025     0.049        40
customer_service      0.000     0.000     0.000        40
    edit_account      0.600     0.075     0.133        40
             pay      0.000     0.000     0.000        30
   payment_issue      0.636     0.350     0.452        40
recover_password      0.744     0.800     0.771        40
  request_refund      0.964     0.675     0.794        40
  track_delivery      0.320     0.200     0.246        40
     track_order      0.654     0.243     

## 11. Sauvegarde du modèle et résumé

In [20]:
# Sauvegarder le modèle SetFit
model.save_pretrained(str(MODEL_DIR))
print(f'Modèle sauvegardé : {MODEL_DIR}')

# Résumé
summary = {
    'model': {
        'backbone':      BACKBONE,
        'num_labels':    NUM_LABELS,
        'train_samples': len(ds_train),
        'mode':          'test_rapide' if NUM_SAMPLES_PER_CLASS != 'all' else 'production',
    },
    'retail_test': {
        'f1_macro':        round(retail_f1, 4),
        'accuracy':        round(retail_acc, 4),
        'mean_confidence': round(float(np.mean(confs_retail)), 4),
        'type':            'in-distribution',
    },
    'external_test': {
        'f1_macro':        round(ext_f1, 4) if ext_texts else None,
        'accuracy':        round(ext_acc, 4) if ext_texts else None,
        'mean_conf_ood':   round(mean_ood_conf, 4) if ext_texts else None,
        'ood_gap':         round(float(np.mean(confs_retail)) - mean_ood_conf, 4) if ext_texts else None,
        'type':            'out-of-distribution',
    },
}

import json
with open(RESULTS_DIR / 'setfit_summary.json', 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('\nRésumé :')
print(json.dumps(summary, indent=2, ensure_ascii=False))


Modèle sauvegardé : D:\conv_nlp_pipeline\models\setfit

Résumé :
{
  "model": {
    "backbone": "sentence-transformers/all-MiniLM-L6-v2",
    "num_labels": 66,
    "train_samples": 528,
    "mode": "test_rapide"
  },
  "retail_test": {
    "f1_macro": 0.0,
    "accuracy": 0.0,
    "mean_confidence": 0.78,
    "type": "in-distribution"
  },
  "external_test": {
    "f1_macro": 0.3333,
    "accuracy": 0.2942,
    "mean_conf_ood": 0.3149,
    "ood_gap": 0.4651,
    "type": "out-of-distribution"
  }
}


## 12. Interprétation & Passage en production

### Ce que vous regardez pour valider la convergence

| Signal | Seuil 'ça marche' | Seuil 'bon' |
|--------|------------------|-------------|
| F1 Retail test (mode test 8ex) | > 0.60 | > 0.75 |
| F1 externe mappé | > 0.25 | > 0.40 |
| Confiance Retail | > 0.75 | > 0.85 |
| Gap OOD | > 0.10 | > 0.20 |

### Si ça converge — passer en production

Changer ces deux lignes :
```python
NUM_SAMPLES_PER_CLASS = 'all'          # utiliser tout le train
NUM_EPOCHS_CONTRASTIVE = 3             # plus d'epochs contrastifs
BACKBONE = 'paraphrase-mpnet-base-v2'  # backbone plus puissant
```

### Comparaison DistilBERT vs SetFit dans le rapport

| Modèle | F1 Retail | F1 externe | Gap OOD | Interprétabilité |
|--------|-----------|------------|---------|------------------|
| DistilBERT | 0.86 | 0.10 (biaisé) | 0.30 | Faible |
| SetFit | ? | ? | ? | Meilleure |

### Installation si manquante
```bash
pip install setfit sentence-transformers
```
